In [ ]:
## link to investments. prioritse source that has investment & capacity. 

In [4]:
import pandas as pd
import sys; sys.path.append("..")
from src.vote_helpers import fill_single_product_lv2, parse_capacity_value
from mongo_client import mongo_client, capacities_collection
from collections import defaultdict

try:
    mongo_client.admin.command("ping")
    print("✅ Connected to MongoDB Atlas!")
except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    raise

df = pd.read_excel("storage/output/filtered_projects_jul16.xlsx")
df['date'] = pd.to_datetime(df['date'], format='%Y-%m')

# fill product_lv2 across multiple NA rows
df = df.groupby("cluster_id").apply(fill_single_product_lv2).reset_index(drop=True)

# normalise capacity values
df["capacity_normalized"] = df["capacity_normalized"].apply(parse_capacity_value)

print(len(df))

print(len(df[(df["phase"] == "unclear")]))

print(len(df[df["status"] == "unclear"]))

print(len(df[df["date"].isna()]))

# drop all cases where phase | status are unclear

df = df[(df["phase"] != "unclear") & (df["status"] != "unclear")].copy()

df = df[~df["date"].isna()].copy()

print(len(df))

✅ Connected to MongoDB Atlas!
345
26
1
47
277


/var/folders/2d/7gfcftkd69s2z741sq98f0sc0000gn/T/ipykernel_44727/3256782809.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("cluster_id").apply(fill_single_product_lv2).reset_index(drop=True)


In [ ]:
import logging

# Remove all handlers associated with the root logger object (to avoid duplicate logs in Jupyter)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Set up logging to file and console
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("votes_log.log", mode='w'),  # opening in write mode (overwriting each time)
        logging.StreamHandler()  # This keeps output in the notebook as well
    ]
)

# Example usage
logging.info("INITIATING LOGGING")
unique_clusters = df["cluster_id"].unique().tolist()
logging.info(f"Votes for {len(unique_clusters)} clusters.")
status_counts = defaultdict(int)

for cluster in unique_clusters:

    logging.info(f"Assessing votes for cluster_id: {cluster}")

    df_cluster = df[df["cluster_id"] == cluster].copy()
    unique_statuses = df_cluster["status"].unique()
    unique_phases = df_cluster["phase"].unique()

    owner = df_cluster["inst_canon"].unique()[0]
    iso2 = df_cluster["iso2"].unique()[0]
    adm1 = df_cluster["adm1"].unique()[0]
    product_lv1 = df_cluster["product_lv1"].unique()[0]

    if "announced" in unique_statuses and len(unique_statuses) <2:
        logging.info("⚪️ ANNOUNCED case")
        status_counts["announced"] += 1

        # only announcement so we ignore expansions 
        df_canon = df_cluster[df_cluster["phase"] != "expansion"]
        if len(df_canon) < 1:
            logging.info("🔴 ERROR - no greenfield phase noted at facility, only expansions. DROPPED.")
            status_counts["announced-dropped-only-expansions"] += 1
            continue

        dt_announce = df_canon["date"].min()
        phase = "greenfield"

        # find row with the latest date & take capacity
        row_latest = df_canon.loc[df_canon["date"].idxmax()]
        capacity_latest = row_latest["capacity_normalized"]
        
        dt_construct = None
        dt_operation = None

    elif "under construction" in unique_statuses and "operational" not in unique_statuses:
        logging.info("🔵 UNDER CONSTRUCTION case")
        status_counts["under-construction"] += 1

        # only under construction so we ignore expansions 
        df_canon = df_cluster[df_cluster["phase"] != "expansion"]
        if len(df_canon) < 1:
            logging.info("🔴 ERROR - no greenfield phase noted at facility, only expansions. DROPPED.")
            status_counts["under-construction-dropped-only-expansions"] += 1
            continue

        dt_announce = df_canon["date"].min()
        status = "under construction"
        phase = "greenfield"

        # Find row with the latest date & take capacity (for under construction cases)
        df_construct = df_canon[df_canon["status"] == "under construction"].copy()
        dt_construct = df_construct["date"].min()
        row_latest_construction = df_construct.loc[df_construct["date"].idxmax()]
        capacity_latest = row_latest_construction["capacity_normalized"]
        
        dt_operation = None

    elif "operational" in unique_statuses:
        logging.info("🟢 OPERATIONAL case")
        status_counts["operational"] += 1

        # cases where there are no expansions
        if "expansion" not in unique_phases:
            logging.info("No expansions sub-case")
            status_counts["operational-no-expansions"] += 1

            # earliest date taken for dt_announce
            dt_announce = df_cluster["date"].min()    #the earliest date mentioned
            status = "operational"
            phase = "greenfield"

            # earliest date where status == under construction taken for dt_construct
            df_construct = df_cluster[df_cluster["status"] == "under construction"].copy()
            dt_construct = df_construct["date"].min()

            # earliest date where status == operational taken for dt_operation
            df_operation = df_cluster[df_cluster["status"] == "operational"].copy()
            dt_operation = df_cluster["date"].min()

            # take operational capacity from most recent article
            row_latest_operation = df_operation.loc[df_operation["date"].idxmax()]
            capacity_latest = row_latest_operation["capacity_normalized"]

        if "expansion" in unique_phases:
            logging.info("With expansions sub-case")
            status_counts["operational-with-expansions"] += 1


    else:
        logging.info("Group not assigned")
        status_counts["unassigned"] += 1

        # only consider expansions AFTER the operational Greenfield row. 

# # final dictionary to insert to Mongo
#     mongo_entry = {
#         "owner": owner,
#         "iso2": iso2,
#         "adm1": adm1,
#         "product_lv1": product_lv1,
#         #"product": product_dict,
#         "dt_announce": dt_announce.strftime("%Y-%m") if pd.notnull(dt_announce) else None,
#         "status": "announced",
#         "phase": phase,
#         "capacity": capacity_latest,
#         "dt_construct": dt_construct.strftime("%Y-%m") if pd.notnull(dt_construct) else None,
#         "dt_operation": dt_operation.strftime("%Y-%m") if pd.notnull(dt_operation) else None
#     }

#     capacities_collection.insert_one(mongo_entry)
#     logging.info("✅ Inserted announced project into MongoDB.")

logging.info("📊 Final Status Counts:")
for status, count in status_counts.items():
    logging.info(f"{status}: {count}")

2025-07-17 17:47:19,620 - INFO - INITIATING LOGGING
2025-07-17 17:47:19,620 - INFO - Votes for 71 clusters.
2025-07-17 17:47:19,621 - INFO - Assessing votes for cluster_id: 2
2025-07-17 17:47:19,622 - INFO - 🔵 UNDER CONSTRUCTION case
2025-07-17 17:47:19,623 - INFO - Assessing votes for cluster_id: 35
2025-07-17 17:47:19,624 - INFO - 🔵 UNDER CONSTRUCTION case
2025-07-17 17:47:19,625 - INFO - Assessing votes for cluster_id: 37
2025-07-17 17:47:19,625 - INFO - 🔵 UNDER CONSTRUCTION case
2025-07-17 17:47:19,626 - INFO - Assessing votes for cluster_id: 39
2025-07-17 17:47:19,626 - INFO - 🔵 UNDER CONSTRUCTION case
2025-07-17 17:47:19,627 - INFO - Assessing votes for cluster_id: 41
2025-07-17 17:47:19,627 - INFO - 🟢 OPERATIONAL case
2025-07-17 17:47:19,628 - INFO - No expansions sub-case
2025-07-17 17:47:19,632 - INFO - Assessing votes for cluster_id: 42
2025-07-17 17:47:19,640 - INFO - 🟢 OPERATIONAL case
2025-07-17 17:47:19,643 - INFO - With expansions sub-case
2025-07-17 17:47:19,644 - INFO 

In [9]:
df[df["cluster_id"] == 49]

,inst_canon,city_key,adm1,adm2,iso2,cluster_id,product_lv1,product_lv2,product,capacity_normalized,capacity_metric_normalized,status,phase,date,article_id
28,microvast gmbh,ludwigsfelde,Brandenburg,NaN,DE,49,battery,module_pack,battery modules,1.5,GWh,operational,greenfield,2021-02-01,67f92164f431fddd61d55af0
29,microvast gmbh,ludwigsfelde,Brandenburg,NaN,DE,49,battery,module_pack,battery modules,6.0,GWh,announced,expansion,2021-02-01,67f92164f431fddd61d55af0


In [14]:
df[df["cluster_id"] == 312]

,inst_canon,city_key,adm1,adm2,iso2,cluster_id,product_lv1,product_lv2,product,capacity_normalized,capacity_metric_normalized,status,phase,date,article_id
133,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,battery cells,12.0,GWh,operational,greenfield,2019-05-01,67f92108f431fddd61d559bc
134,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,battery cells,30.0,GWh,announced,expansion,2019-05-01,67f92108f431fddd61d559bc
135,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,battery cells,16.0,GWh,announced,expansion,2019-09-01,67f92112f431fddd61d559dd
136,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,solid-state lithium-metal batteries,1.0,GWh,announced,greenfield,2021-05-01,67f92163f431fddd61d55aed
137,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,solid-state lithium-metal batteries,21.0,GWh,announced,expansion,2021-05-01,67f92163f431fddd61d55aed
138,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,unit cells,20.0,GWh,announced,greenfield,2021-12-01,67f921a2f431fddd61d55bc1
139,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,unit cells,40.0,GWh,announced,expansion,2021-12-01,67f921a2f431fddd61d55bc1
142,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,battery cells,40.0,GWh,announced,greenfield,2023-10-01,67f92299f431fddd61d55ef2
144,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,unified cell,20.0,GWh,operational,greenfield,2024-09-01,67f92324f431fddd61d560b9
145,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,battery cells,40.0,GWh,under construction,greenfield,2024-12-01,67f92310f431fddd61d56077
